In [232]:
import duckdb

con = duckdb.connect("health_enforcement.duckdb")
con.execute("SHOW TABLES").df()


,name
0,dim_district_office
1,dim_fac_code
2,enriched_mart
3,fact_enforcement_actions
4,fact_ltc_narratives


In [233]:
df_enforcement=con.execute("SELECT * FROM fact_enforcement_actions").df()
df_enforcement

,FACID,FACILITY_NAME,LTC,FAC_TYPE_CODE,FAC_FDR,DISTRICT_OFFICE,PENALTY_ISSUE_DATE,PENALTY_NUMBER,DISPOSITION,PENALTY_TYPE,...,TOTAL_COLLECTED_AMOUNT,TOTAL_BALANCE_DUE,ACLAIMS_VISIT_NUMBER,EVENTID,DEATH_RELATED,INTAKEID_ALL,PRIORITY_ALL,SFY,HIGHEST_PRIORITY,COMPLAINT_COUNT
0,10000060,"SEAVIEW REHABILITATION & WELLNESS CENTER, LP",LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-06,10001426,Closed,Citation,...,650.0,0.0,10010610.0,A99999,N,NaN,NaN,SFY1999-00,NaN,0
1,10000061,PALM DRIVE NURSING AND REHABILITATION CENTER,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-26,10001435,Closed,Citation,...,7000.0,0.0,10010652.0,A99999,N,NaN,NaN,SFY1999-00,NaN,0
2,10000043,PARK VIEW POST ACUTE,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-26,10001436,Closed,Citation,...,1500.0,0.0,10010654.0,A99999,N,NaN,NaN,SFY1999-00,NaN,0
3,10000043,PARK VIEW POST ACUTE,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-27,10001437,Closed,Citation,...,1000.0,0.0,10010657.0,A99999,N,NaN,NaN,SFY1999-00,NaN,0
4,10000062,"SHERWOOD OAKS POST ACUTE CARE, LLC",LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-28,10001438,Closed,Citation,...,650.0,0.0,10010659.0,A99999,N,NaN,NaN,SFY1999-00,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20545,930000095,MEMORIALCARE LONG BEACH MEDICAL CENTER,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-08-15,930019771,Open,Administrative Penalty,...,0.0,20000.0,NaN,K5TO11,N,CA00810014,D,SFY2023-24,D,1
20546,930000129,VALLEY PRESBYTERIAN HOSPITAL,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-10-24,930019974,Closed,Administrative Penalty,...,93750.0,0.0,NaN,1CJ911,N,CA00777415,C,SFY2023-24,C,1
20547,930000129,VALLEY PRESBYTERIAN HOSPITAL,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-04-11,930019977,Closed,Administrative Penalty,...,187500.0,0.0,NaN,2TCN11,N,CA00799417,C,SFY2022-23,C,1
20548,930000133,PACIFICA HOSPITAL OF THE VALLEY,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2024-04-24,930020061,Open,Administrative Penalty,...,0.0,3636.0,NaN,DELK11,N,CA00880498,B,SFY2023-24,B,1


In [236]:
# 1. Row counts — nothing should explode or disappear
print("Row Counts")
print(f"fact_enforcement_actions: {con.execute('SELECT COUNT(*) FROM fact_enforcement_actions').fetchone()[0]}")
print(f"fact_ltc_narratives:      {con.execute('SELECT COUNT(*) FROM fact_ltc_narratives').fetchone()[0]}")
print(f"enriched_mart:            {con.execute('SELECT COUNT(*) FROM enriched_mart').fetchone()[0]}")

# 2. enriched_mart should match fact_enforcement_actions exactly
print("mart and fact_enforcement")
mart_rows = con.execute("SELECT COUNT(*) FROM enriched_mart").fetchone()[0]
fact_rows = con.execute("SELECT COUNT(*) FROM fact_enforcement_actions").fetchone()[0]
print(f"Mart rows == Fact rows: {mart_rows == fact_rows} ({mart_rows} vs {fact_rows})")

# 4. Nulls in joined columns — count mismatches
print("Mismatches")
print(con.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(CASE WHEN FACILITY_TYPE_DESC IS NULL THEN 1 ELSE 0 END) AS missing_fac_type,
        SUM(CASE WHEN TOP_KEYWORDS IS NULL THEN 1 ELSE 0 END)       AS missing_keywords
    FROM enriched_mart
""").df())

# 5. Spot check a known facility
print("Checkin Fac")
print(con.execute("""
    SELECT FACID, FACILITY_NAME, FACILITY_TYPE_DESC, 
           TOP_KEYWORDS
    FROM enriched_mart
    WHERE TOP_KEYWORDS IS NOT NULL
    LIMIT 5
""").df())

Row Counts
fact_enforcement_actions: 20550
fact_ltc_narratives:      2885
enriched_mart:            20550
mart and fact_enforcement
Mart rows == Fact rows: True (20550 vs 20550)
Mismatches
   total_rows  missing_fac_type  missing_keywords
0       20550               1.0           17668.0
Checkin Fac
       FACID                           FACILITY_NAME  \
0   20000132               FREMONT HEALTHCARE CENTER   
1   20000054  PRINCETON MANOR HEALTHCARE CENTER, LLC   
2  140000051    LA CASA VIA TRANSITIONAL CARE CENTER   
3   20000026   BAY VIEW REHABILITATION HOSPITAL, LLC   
4  140000105                    LONE TREE POST ACUTE   

                                  FACILITY_TYPE_DESC  \
0  A health facility that provides skilled nursin...   
1  A health facility that provides skilled nursin...   
2  A health facility that provides skilled nursin...   
3  A health facility that provides skilled nursin...   
4  A health facility that provides skilled nursin...   

                   TOP_KE

In [231]:
con.close()

In [184]:
# print(df_facility_types[df_facility_types.duplicated(subset='VALUE', keep=False)][['VALUE', 'DESCRIPTION']].sort_values('VALUE'))

Empty DataFrame
Columns: [VALUE, DESCRIPTION]
Index: []
